In [ ]:
#!/usr/bin/env python3
"""Cell 0 of the generated runner notebook — copied byte for byte.

`jobfolder.build_notebook()` copies this file's bytes into the notebook's
first cell with ZERO interpolation. The same bytes serve every job because
every fact this cell needs comes from `run-config.json`, read at runtime,
never baked in at generation time. Ten responsibilities, in order:

1. read `run-config.json`; validate schema/version
2. sparse-clone the pinned commit: `git init`, `remote add`,
   `sparse-checkout set <clonePaths>`, `fetch --depth 1 origin <commit>`,
   `checkout FETCH_HEAD`. When the job staged a repository credential
   beside this cell (`git-askpass.sh`, S3), every clone call carries
   `GIT_ASKPASS` pointing at it, a non-`https://` `repo.url` refuses
   before the first git call, and both material files are deleted in a
   `finally` immediately after the clone attempt
3. `sys.path.insert(0, <clone>/src)`
4. install the declared build: `sys.executable -m pip install`, list
   argv, against `run-config.json`'s additive `environment.install`
   (Decision 3). This runs BEFORE responsibility 5, never after —
   responsibility 5 imports the declared modules, and those modules
   import the tensor library this step installs; installing after that
   point would be too late. An older, undeclared config carries no
   `environment.install` block and this step is a no-op. Honest cost:
   this installs on every kernel that runs a job, adding minutes to each
   one — a campaign of thirty shards pays that cost thirty times.
   Determinism (the arriving capability verified against what was
   actually installed, never assumed) was chosen over the cheaper
   deferral of trusting a pre-baked image
5. import each declared module and assert its `__file__` resolves under
   the clone's own `src` — the "pip-installed copy" refusal: a module
   importable from somewhere ELSE already on `sys.path` would silently
   run against code this job never pinned a commit for. A declared
   NOTEBOOK gets the same proof in the only form a notebook can take
   (`verify_notebooks_under_clone`): the file is where the pinned sparse
   checkout should have put it, and it names a kernel
6. detect hardware — `torch` not importable IS "hardware missing"; no
   silent CPU fallback, or this refusal could never actually fire
7. write `bootstrap.json`: commit, config, detected environment, resolved
   imports and resolved notebooks
8. any of config / code / hardware missing raises `SystemExit` on the
   spot, so cell 1 never runs against a half-prepared runtime
9. the notebook-executor gate, when a notebook is declared: `nbformat`,
   `nbclient` and `jupyter_client` must be importable and the notebook's
   own kernel must resolve to an installed kernelspec. Probed, never
   assumed — this cell installs only what the target declared, and the
   remedy a refusal names is that same declaration. Runs AFTER
   responsibility 7 for the same reason the accelerator gate does
10. the accelerator gate: the arriving capability must appear in the
   installed arch list, and any declared `accelerator.architectures`
   must be covered by that same installed list. This runs AFTER
   responsibility 7, never before — a refusal whose evidence was never
   written is unreadable no matter how early it fires, so `bootstrap.json`
   already carries the arriving device, the torch build and the arch
   list the verdict was computed from by the time this refuses
11. no service name anywhere, ever

Importable and independently testable: every responsibility above is a
plain function, and `bootstrap()` composes them. Nothing runs at import
time — the orchestrating call sits behind `if __name__ == "__main__":`,
the state a notebook cell's own top-level code runs in (a Jupyter
kernel's namespace has `__name__ == "__main__"`), so importing this file
under a different module name never fires it. That is what lets the forge
suite drive `bootstrap()` and every helper directly against fake configs,
in-process — the whole justification `jobfolder.py` gives for shipping
this as a byte-for-byte copy rather than a per-job embedded prose cell.

Every git call goes through `_run_git()`, the single composition point:
`shell=False`, list argv, a PATH-only env allowlist, an explicit timeout,
non-zero exit is a refusal.

Run with any Python 3.10+ (stdlib-only):
    python3 -m unittest tests.test_remote_execution
"""
from __future__ import annotations

import importlib
import json
import os
import subprocess
import sys
from pathlib import Path
from typing import Any, Callable, Mapping, Sequence


class BootstrapError(Exception):
    """A refusal: the run configuration, the declared code, or this
    runtime's hardware is missing or invalid. `bootstrap()` is the only
    place that turns one of these into `SystemExit`.
    """


class AcceleratorError(BootstrapError):
    """Responsibility 8's refusal: the arriving accelerator cannot run
    this build's kernels, or the declared architectures are not covered
    by the torch build actually installed. A subclass of
    `BootstrapError` so `bootstrap()`'s existing `except BootstrapError`
    converts it to `SystemExit` the same way as every other refusal —
    no second exception-handling path to keep in step.
    """


CONFIG_FILENAME = "run-config.json"
BOOTSTRAP_OUTPUT_FILENAME = "bootstrap.json"
CLONE_DIRNAME = "clone"
SRC_DIRNAME = "src"
RUN_CONFIG_SCHEMA_VERSION = 1
REQUIRED_RUN_CONFIG_FIELDS = ("schemaVersion", "commit", "repo", "clonePaths", "run")

# The whole allowlist a git subprocess's environment is built from — never
# `os.environ` forwarded wholesale, the same restraint every other
# subprocess call in this skill applies to its own child environment.
GIT_ENV_ALLOWLIST = ("PATH",)
GIT_TIMEOUT_SECONDS = 120.0

# The credential material a credentialed submission stages beside the
# executor (S3): an askpass script and the token file it reads. The names
# are declared in THREE files by convention — the backend adapter (the
# uploader and the fetch-time refusal), the executor asset (chmod,
# deletion, manifest exclusion) and this one (the clone's own use and
# deletion) — deliberately not shared by import: no module above the seam
# may name a backend, and the asset files are byte-copies, never importers
# of each other. `git-askpass.sh` prints the token file's bytes for
# whichever prompt git asks; HTTPS token auth on the hosts this skill
# targets accepts the token as the username with any password.
REPO_CREDENTIAL_ASKPASS_FILENAME = "git-askpass.sh"
REPO_CREDENTIAL_TOKEN_FILENAME = "git-askpass-token"


def _config_path(base_dir: str | Path | None) -> Path:
    base = Path(base_dir) if base_dir is not None else Path.cwd()
    return base / CONFIG_FILENAME


def load_run_config(base_dir: str | Path | None = None) -> dict:
    """Read `run-config.json` beside the notebook (or `base_dir`, for a
    test) and validate its schema/version — responsibility 1.
    """
    path = _config_path(base_dir)
    if not path.is_file():
        raise BootstrapError(
            f"config missing: {path} does not exist; nothing to bootstrap from"
        )
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        raise BootstrapError(f"config missing: {path} is not valid JSON: {exc}") from exc
    _validate_run_config(payload)
    return payload


def _validate_run_config(run_config: object) -> None:
    if not isinstance(run_config, dict):
        raise BootstrapError("config missing: run-config.json must decode to a JSON object")
    missing = [f for f in REQUIRED_RUN_CONFIG_FIELDS if f not in run_config]
    if missing:
        raise BootstrapError(
            f"config missing: run-config.json missing required fields: {missing}"
        )
    if run_config.get("schemaVersion") != RUN_CONFIG_SCHEMA_VERSION:
        raise BootstrapError(
            f"config missing: run-config.json declares schemaVersion "
            f"{run_config.get('schemaVersion')!r}; this bootstrap reads only "
            f"{RUN_CONFIG_SCHEMA_VERSION}"
        )
    run_block = run_config.get("run")
    if not isinstance(run_block, dict):
        raise BootstrapError(
            "config missing: run-config.json's 'run' block must be an object"
        )
    # Exactly one of the two shapes, and this cell refuses rather than
    # guesses at a block that declares neither or both. The refusal has to
    # exist HERE and not only in the generator: this cell reads whatever
    # `run-config.json` the kernel was actually handed, and a config that
    # acquired a shapeless `run` block any other way (hand-edited, staged
    # by an older generator) reaches this file and nothing else.
    has_callable = bool(run_block.get("module"))
    has_notebook = bool(run_block.get("notebook"))
    if has_callable and has_notebook:
        raise BootstrapError(
            "config missing: run-config.json's 'run' block declares BOTH a "
            "'module' and a 'notebook'; a run has exactly one shape"
        )
    if not (has_callable or has_notebook):
        raise BootstrapError(
            "config missing: run-config.json's 'run' block must declare "
            "either a 'module' or a 'notebook'"
        )


def _run_git(
    args: Sequence[str],
    *,
    cwd: str | Path,
    timeout: float = GIT_TIMEOUT_SECONDS,
    extra_env: Mapping[str, str] | None = None,
) -> subprocess.CompletedProcess:
    """The single composition point for every git invocation this cell
    makes. `shell=False` with a list argv means a value carrying shell
    metacharacters reaches `argv` as one element and is never evaluated —
    no shell is ever invoked to evaluate it. The environment is built from
    `GIT_ENV_ALLOWLIST` alone, never this process's own `os.environ`
    forwarded wholesale. A non-zero exit or an expired timeout raises
    `BootstrapError` rather than being silently ignored.

    `extra_env` (S3) is a caller-supplied mapping merged into the child's
    environment AFTER the allowlist — the credential-aware clone's
    `GIT_ASKPASS`, and never anything else. It is a separate channel from
    `GIT_ENV_ALLOWLIST` on purpose: the allowlist stays `("PATH",)`, and
    the askpass path is an explicit per-call decision rather than a
    widening of it. `None` (every credentialless clone) means the child's
    environment is built exactly as it was before this parameter existed.
    """
    argv = ["git", *args]
    env = {name: os.environ[name] for name in GIT_ENV_ALLOWLIST if name in os.environ}
    if extra_env:
        env.update(extra_env)
    try:
        result = subprocess.run(
            argv,
            shell=False,
            cwd=str(cwd),
            env=env,
            capture_output=True,
            text=True,
            timeout=timeout,
        )
    except subprocess.TimeoutExpired as exc:
        raise BootstrapError(f"git {' '.join(args)} timed out after {timeout}s") from exc
    except OSError as exc:
        raise BootstrapError(f"could not run git: {exc}") from exc
    if result.returncode != 0:
        raise BootstrapError(
            f"git {' '.join(args)} exited {result.returncode}: {result.stderr.strip()}"
        )
    return result


def clone_repo(
    run_config: Mapping[str, Any],
    clone_dir: str | Path,
    *,
    askpass: str | Path | None = None,
) -> Path:
    """Sparse-clone the declared repo at the pinned commit — responsibility
    2 — every step through `_run_git()` alone: `init`, `remote add`,
    `sparse-checkout set <clonePaths>`, `fetch --depth 1 origin <commit>`,
    `checkout FETCH_HEAD`.

    `askpass` (S3) is the absolute path to the staged `git-askpass.sh`
    when — and only when — the submission carried a repository credential.
    With it, every git call this function makes carries
    `GIT_ASKPASS=<that path>` through `_run_git()`'s `extra_env`, and
    `run-config.json`'s `repo.url` must be `https://` — refused BEFORE the
    first git call otherwise, because a hand-edited config can bypass
    generation and a credential can only travel over TLS (the runner has
    no SSH agent and no key). The two material files are deleted in a
    `finally` immediately after the clone attempt, success or failure:
    that is D6's deletion point — before anything else in cell 0 proceeds
    — and a failed unlink is a printed `warning:` line, never a raise,
    because the clone's own outcome must not be masked and the executor's
    own fallback deletion plus its manifest exclusion already bound the
    residue. `None` (every credentialless job, and every run of a backend
    that stages no credential material) is byte-identical to before this
    parameter existed: no `GIT_ASKPASS`, no deletion, no URL gate.
    """
    clone_dir = Path(clone_dir)
    clone_dir.mkdir(parents=True, exist_ok=True)
    repo = run_config["repo"]
    commit = run_config["commit"]
    clone_paths = list(run_config["clonePaths"])

    askpass_path: Path | None = Path(askpass) if askpass is not None else None
    if askpass_path is not None:
        url = str(repo.get("url", ""))
        if not url.startswith("https://"):
            raise BootstrapError(
                f"repo credential refused: run-config.json's repo.url "
                f"{url!r} is not an https:// URL, and a staged repository "
                "credential can only travel over HTTPS — the runner's clone "
                "has no SSH agent and no key. Refusing before any git call."
            )
        clone_env: Mapping[str, str] | None = {
            "GIT_ASKPASS": str(askpass_path.resolve())
        }
    else:
        clone_env = None

    def _git(args: Sequence[str]) -> subprocess.CompletedProcess:
        return _run_git(args, cwd=clone_dir, extra_env=clone_env)

    try:
        _git(["init"])
        _git(["remote", "add", "origin", repo["url"]])
        _git(["sparse-checkout", "set", *clone_paths])
        _git(["fetch", "--depth", "1", "origin", commit])
        _git(["checkout", "FETCH_HEAD"])
    finally:
        if askpass_path is not None:
            for material in (
                askpass_path,
                askpass_path.parent / REPO_CREDENTIAL_TOKEN_FILENAME,
            ):
                try:
                    material.unlink()
                except FileNotFoundError:
                    pass
                except OSError as exc:
                    print(
                        f"warning: could not delete credential material "
                        f"{material}: {exc}"
                    )
    return clone_dir


def add_clone_to_path(clone_dir: str | Path) -> Path:
    """`sys.path.insert(0, <clone>/src)` — responsibility 3."""
    src_dir = (Path(clone_dir) / SRC_DIRNAME).resolve()
    sys.path.insert(0, str(src_dir))
    return src_dir


# The whole allowlist a pip subprocess's environment is built from —
# never `os.environ` forwarded wholesale, the same restraint
# `GIT_ENV_ALLOWLIST` above already applies to its own child environment.
# A separate constant, deliberately: this cell's pip invocation and its
# git invocations are different subprocess families, and each holds this
# discipline on its own rather than sharing one name that could drift for
# the wrong reason.
PIP_ENV_ALLOWLIST = ("PATH",)
PIP_INSTALL_TIMEOUT_SECONDS = 600.0


def _validate_requirement_specifier(spec: str) -> str:
    """A requirement specifier is data, never a flag. `pip install` reads
    each positional argv element as either a package specifier or, when
    it begins with `-`, an option of its own — a declared specifier
    shaped like `--index-url https://evil.invalid` would occupy the same
    argv position as a real package name and silently redirect the whole
    install to an attacker-controlled index. Refusing any specifier
    beginning with `-` is what keeps a declared `requirements` list
    unable to smuggle a flag this way. Nothing else about the string is
    inspected: shell metacharacters elsewhere in a specifier reach pip's
    argv as inert data, the same `shell=False`, list-argv guarantee
    `_run_git()` already gives every value this cell passes to git.
    """
    if not isinstance(spec, str) or not spec or spec.startswith("-"):
        raise BootstrapError(
            f"environment.install refuses requirement specifier {spec!r}: "
            "a specifier beginning with '-' would be read by pip as an "
            "option, not a package to install"
        )
    return spec


def _run_pip(args: Sequence[str], *, timeout: float = PIP_INSTALL_TIMEOUT_SECONDS) -> None:
    """The single composition point for the one pip invocation this cell
    makes — the same discipline `_run_git()` already holds: `shell=False`
    with a list argv, so a value carrying shell metacharacters reaches
    argv as one element and is never evaluated by a shell; the
    environment built from `PIP_ENV_ALLOWLIST` alone, never this
    process's own `os.environ` forwarded wholesale; an explicit timeout;
    a non-zero exit is a refusal, never silently ignored.
    """
    argv = [sys.executable, "-m", "pip", *args]
    env = {name: os.environ[name] for name in PIP_ENV_ALLOWLIST if name in os.environ}
    try:
        result = subprocess.run(
            argv,
            shell=False,
            env=env,
            capture_output=True,
            text=True,
            timeout=timeout,
        )
    except subprocess.TimeoutExpired as exc:
        raise BootstrapError(
            f"environment install timed out after {timeout}s running pip "
            f"{' '.join(args)}"
        ) from exc
    except OSError as exc:
        raise BootstrapError(f"could not run pip: {exc}") from exc
    if result.returncode != 0:
        raise BootstrapError(
            f"pip {' '.join(args)} exited {result.returncode}: "
            f"{result.stderr.strip()}"
        )


def install_environment(run_config: Mapping[str, Any]) -> None:
    """Responsibility 4: install the dual-architecture torch build (or
    whatever else the target declares) BEFORE responsibility 5 imports
    the target's own modules — those imports pull in the tensor library
    this step installs, so installing after that point would be too
    late.

    Reads `run-config.json`'s additive `environment.install:
    {requirements[], indexUrl}` (Decision 3). Additive: an older config
    with no `environment` block, or no `install` block within it,
    installs nothing at all — this is a no-op, exactly as this cell
    behaved before the field existed. This module names only those two
    fields and never a package: it does not know which tensor-library
    build a target needs, only that the target declared one.

    Every requirement specifier is validated by
    `_validate_requirement_specifier()` before any argv is built, and the
    whole install runs through `_run_pip()`, the single composition point
    matching `_run_git()`'s own discipline: `sys.executable -m pip
    install`, with `--index-url <indexUrl>` first when declared, then
    each requirement its own argv element — never one opaque command
    string.
    """
    environment = run_config.get("environment")
    install = environment.get("install") if isinstance(environment, Mapping) else None
    if not isinstance(install, Mapping):
        return
    requirements = [
        _validate_requirement_specifier(spec)
        for spec in (install.get("requirements") or [])
    ]
    if not requirements:
        raise BootstrapError(
            "config missing: environment.install declares no requirements "
            "to install"
        )
    index_url = install.get("indexUrl")
    args = ["install"]
    if index_url:
        args += ["--index-url", str(index_url)]
    args += requirements
    _run_pip(args)


def declared_modules(run_config: Mapping[str, Any]) -> list[str]:
    """Every module `run-config.json` names as an entry point: the normal
    `run.module`, plus `run.smoke.module` when present — both get the
    same `__file__`-under-clone proof at bootstrap time, since either one
    could be the module cell 1 actually calls.
    """
    run_block = run_config["run"]
    modules = []
    if run_block.get("module"):
        modules.append(run_block["module"])
    smoke = run_block.get("smoke")
    if isinstance(smoke, dict) and smoke.get("module"):
        modules.append(smoke["module"])
    return modules


def declared_notebooks(run_config: Mapping[str, Any]) -> list[str]:
    """Every notebook `run-config.json` names as an entry point: the normal
    `run.notebook`, plus `run.smoke.notebook` when present — the notebook
    shape's exact counterpart to `declared_modules()` above, and read the
    same way, since either one could be what cell 1 actually runs.
    """
    run_block = run_config["run"]
    notebooks = []
    if run_block.get("notebook"):
        notebooks.append(run_block["notebook"])
    smoke = run_block.get("smoke")
    if isinstance(smoke, dict) and smoke.get("notebook"):
        notebooks.append(smoke["notebook"])
    return notebooks


def verify_imports_under_clone(
    modules: Sequence[str],
    src_dir: str | Path,
    *,
    import_module: Callable[[str], Any] = importlib.import_module,
) -> dict[str, str]:
    """Import each declared module and assert its `__file__` resolves
    under the clone's own `src` — responsibility 5, the "pip-installed
    copy" refusal. `.resolve()` on both sides: a temp path traversing a
    `/var` -> `/private/var`-style symlink must compare equal either way.
    """
    resolved_src = Path(src_dir).resolve()
    verified: dict[str, str] = {}
    for name in modules:
        try:
            module = import_module(name)
        except ImportError as exc:
            raise BootstrapError(
                f"code missing: declared module {name!r} could not be imported: {exc}"
            ) from exc
        module_file = getattr(module, "__file__", None)
        if not module_file:
            raise BootstrapError(
                f"code missing: module {name!r} carries no __file__; its location "
                "cannot be verified"
            )
        resolved_file = Path(module_file).resolve()
        try:
            resolved_file.relative_to(resolved_src)
        except ValueError:
            raise BootstrapError(
                f"code missing: module {name!r} resolved to {resolved_file}, "
                f"outside the clone's own src at {resolved_src} — refusing the "
                "'pip-installed copy' case"
            )
        verified[name] = str(resolved_file)
    return verified


DEFAULT_KERNEL_NAME = "python3"

# The three modules a notebook run needs on the worker, and the whole of
# what `check_notebook_executor()` probes for. `nbformat` reads and writes
# the notebook, `nbclient` executes it, and `jupyter_client` is what
# resolves a kernel NAME to an installed kernel — the third one is not
# redundant with the other two: a runtime can have both readers and no
# kernelspec at all, which is exactly the shape of the failure that has
# already cost this skill a real push.
NOTEBOOK_EXECUTOR_MODULES = ("nbformat", "nbclient", "jupyter_client")


def verify_notebooks_under_clone(
    notebooks: Sequence[str], clone_dir: str | Path
) -> dict[str, dict]:
    """Prove every declared notebook actually ARRIVED in the clone, and
    read the kernel each one asks for — the notebook shape's exact
    counterpart to `verify_imports_under_clone()`, and it exists for the
    same reason that one does.

    A module gets its `__file__` checked because an importable copy from
    somewhere else on `sys.path` would silently run unpinned code. A
    notebook cannot be imported at all, so the equivalent proof is that
    the file is where the pinned checkout should have put it. `git
    sparse-checkout` reports NOTHING for a path its declared patterns do
    not cover, so the failure this refuses against is silent by
    construction: the clone succeeds, the kernel starts, and the run dies
    at the moment it goes looking for the notebook — with the quota
    already spent.

    The kernel name is read from each notebook's own
    `metadata.kernelspec.name`, defaulting to `DEFAULT_KERNEL_NAME` when
    the notebook declares none. It is read HERE rather than in
    `check_notebook_executor()` so the capability gate is handed a fact
    off the pinned notebook instead of re-opening the file to guess one.
    """
    resolved_clone = Path(clone_dir).resolve()
    verified: dict[str, dict] = {}
    for name in notebooks:
        candidate = (resolved_clone / name).resolve()
        try:
            candidate.relative_to(resolved_clone)
        except ValueError:
            raise BootstrapError(
                f"code missing: declared notebook {name!r} resolves to "
                f"{candidate}, outside the clone at {resolved_clone}"
            ) from None
        if not candidate.is_file():
            raise BootstrapError(
                f"code missing: declared notebook {name!r} is not at "
                f"{candidate}; the sparse checkout delivered nothing for it, "
                "which it does silently for any path the declared clone "
                "paths do not cover"
            )
        try:
            payload = json.loads(candidate.read_text(encoding="utf-8"))
        except (OSError, UnicodeDecodeError, json.JSONDecodeError) as exc:
            raise BootstrapError(
                f"code missing: declared notebook {name!r} at {candidate} is "
                f"not readable as a notebook: {exc}"
            ) from exc
        metadata = payload.get("metadata") if isinstance(payload, dict) else None
        kernelspec = metadata.get("kernelspec") if isinstance(metadata, dict) else None
        kernel = None
        if isinstance(kernelspec, dict):
            kernel = kernelspec.get("name")
        verified[name] = {
            "path": str(candidate),
            "kernel": str(kernel) if kernel else DEFAULT_KERNEL_NAME,
        }
    return verified


def check_notebook_executor(
    notebooks: Mapping[str, dict],
    *,
    import_module: Callable[[str], Any] = importlib.import_module,
) -> None:
    """The notebook-executor gate: refuse, here, when this runtime cannot
    execute a notebook — never after the run is already under way.

    This cell installs only what `run-config.json`'s `environment.install`
    declares, and nothing in this skill can know what a given worker image
    ships. So the mechanism is not ASSUMED to be present: it is probed,
    and its absence is a refusal that names the remedy, which is the
    target declaring it in `environment.install` like any other
    dependency. Responsibility 4 runs before this, so a declared install
    is what this gate then measures.

    Three probes, because three different absences produce the same
    unusable runtime:

    - `nbformat`/`nbclient` — the reader and the executor themselves.
    - `jupyter_client` and an actual kernelspec for each notebook's own
      declared kernel. A missing kernelspec is not a hypothetical: a
      notebook with no resolvable kernel made the service's own runner
      refuse before a single cell executed, and that was discovered only
      after a real push with the quota already gone. The same failure,
      one layer in, is a notebook this cell hands to `nbclient` that
      names a kernel nothing here can start.

    A job that declares no notebook passes through untouched — no import
    is attempted, and a callable-shaped run keeps behaving exactly as it
    did before any of this existed.
    """
    if not notebooks:
        return
    for module_name in NOTEBOOK_EXECUTOR_MODULES:
        try:
            import_module(module_name)
        except ImportError as exc:
            raise BootstrapError(
                f"notebook executor missing: {module_name!r} is not importable "
                f"in this runtime, and this job declares a notebook run "
                f"({sorted(notebooks)}). Declare it in run-config.json's "
                f"environment.install (--environment-requirement) so it is "
                f"installed before this cell asks for it: {exc}"
            ) from exc
    kernelspec_module = import_module("jupyter_client.kernelspec")
    manager = kernelspec_module.KernelSpecManager()
    for name, info in sorted(notebooks.items()):
        kernel = info.get("kernel") or DEFAULT_KERNEL_NAME
        try:
            manager.get_kernel_spec(kernel)
        except Exception as exc:  # noqa: BLE001 - any resolution failure is the refusal
            raise BootstrapError(
                f"notebook executor missing: notebook {name!r} declares kernel "
                f"{kernel!r} and this runtime has no kernelspec for it "
                f"({type(exc).__name__}: {exc}); the notebook would be handed "
                "to an executor that cannot start a kernel for it"
            ) from exc


def _capability_to_arch(capability: tuple[int, int]) -> str:
    """`torch.cuda.get_device_capability()`'s `(major, minor)` pair,
    formatted the same way `torch.cuda.get_arch_list()` names its own
    entries (`sm_60`, `sm_75`, ...) — the one shared vocabulary the
    accelerator gate compares against.
    """
    major, minor = capability
    return f"sm_{major}{minor}"


def detect_hardware(
    *, import_module: Callable[[str], Any] = importlib.import_module
) -> dict[str, Any]:
    """Hardware detection — responsibility 6. `torch` not importable IS
    the refusal mapping for "hardware missing": no silent CPU fallback,
    because a silent fallback would mean this refusal branch could never
    actually fire.

    Beside `device` and `torch`, this also captures `archList` — the
    architectures THIS INSTALLED torch build actually ships kernels for
    (`torch.cuda.get_arch_list()`) — and `capability`, the arriving
    device's own capability formatted the same way. Both are what
    responsibility 9's accelerator gate compares; neither requires the
    declared `accelerator` block to exist, since the arch list a build
    installs is a fact about that build, not about what any job declared.
    A runtime with no CUDA device carries nothing to compare: `archList`
    is `[]` and `capability` is `None`.
    """
    try:
        torch = import_module("torch")
    except ImportError as exc:
        raise BootstrapError(
            "hardware missing: torch is not importable in this runtime"
        ) from exc
    cuda_available = bool(torch.cuda.is_available())
    device = {
        "kind": "cuda" if cuda_available else "cpu",
        "name": torch.cuda.get_device_name(0) if cuda_available else "cpu",
    }
    if cuda_available:
        arch_list = list(torch.cuda.get_arch_list())
        capability = _capability_to_arch(torch.cuda.get_device_capability(0))
    else:
        arch_list = []
        capability = None
    return {
        "device": device,
        "torch": str(torch.__version__),
        "archList": arch_list,
        "capability": capability,
    }


def write_bootstrap_output(
    base_dir: str | Path | None,
    *,
    commit: str,
    run_config: Mapping[str, Any],
    environment: Mapping[str, Any],
    imports: Mapping[str, str],
    notebooks: Mapping[str, dict] | None = None,
) -> Path:
    """`bootstrap.json` — responsibility 7: the commit, the config, the
    detected environment, and the resolved import locations responsibility
    4 just proved.

    `notebooks` carries the same proof for the notebook shape — each
    declared notebook's resolved location inside the clone and the kernel
    it asks for — and is written for the same reason `imports` is: the
    notebook-executor gate below refuses AFTER this file exists, so its
    evidence (which notebook, which kernel) is already readable on disk
    when it fires. Always present, `{}` for a callable-shaped run, never
    absent — an absent key and an empty one are different facts and only
    one of them is true here.
    """
    base = Path(base_dir) if base_dir is not None else Path.cwd()
    payload = {
        "commit": commit,
        "config": dict(run_config),
        "environment": dict(environment),
        "imports": dict(imports),
        "notebooks": dict(notebooks or {}),
    }
    path = base / BOOTSTRAP_OUTPUT_FILENAME
    path.write_text(json.dumps(payload, sort_keys=True, indent=2), encoding="utf-8")
    return path


def check_accelerator(run_config: Mapping[str, Any], environment: Mapping[str, Any]) -> None:
    """The accelerator gate — responsibility 9, run only AFTER
    `write_bootstrap_output()` (see `bootstrap()` below). Two assertions:

    1. the arriving `capability` must appear in the INSTALLED `archList`
       — the physics check: can this build run on this card at all. It
       needs no declaration and runs whenever a capability was detected
       (a CPU-only runtime has none, and nothing to refuse here).
    2. the declared `accelerator.architectures`, when `run-config.json`
       declares an `accelerator` block, must be covered by that same
       installed `archList` — this is how the dual-architecture torch
       build gets VERIFIED rather than assumed. An older, undeclared
       config carries no `accelerator` block and this assertion is
       skipped entirely: additive, `schemaVersion` stays 1.
    """
    capability = environment.get("capability")
    arch_list = list(environment.get("archList") or [])
    if capability is not None and capability not in arch_list:
        raise AcceleratorError(
            f"accelerator mismatch: this runtime's capability {capability!r} "
            f"is not in the installed torch build's arch list {arch_list!r}; "
            "training would fail with no kernel image for this device"
        )
    accelerator = run_config.get("accelerator")
    if isinstance(accelerator, Mapping):
        declared = list(accelerator.get("architectures") or [])
        uncovered = [arch for arch in declared if arch not in arch_list]
        if uncovered:
            raise AcceleratorError(
                f"accelerator mismatch: declared architectures {uncovered} "
                f"are not covered by the installed torch build's arch list "
                f"{arch_list!r}; the dual-architecture build was assumed, "
                "never verified"
            )


def bootstrap(
    base_dir: str | Path | None = None,
    *,
    hardware_import: Callable[[str], Any] = importlib.import_module,
) -> dict[str, Any]:
    """The whole of cell 0, in the fixed order the design pins:
    config -> clone -> `sys.path` -> install -> imports -> notebooks ->
    hardware -> `bootstrap.json` -> the notebook-executor gate -> the
    accelerator gate.

    The two gates sit together after the write, and the notebook one is
    first only because it is the cheaper question. Both are runtime
    CAPABILITY questions — can this runtime execute what was declared —
    as against `verify_imports_under_clone()`/`verify_notebooks_under_
    clone()` above them, which ask whether the pinned code ARRIVED and
    therefore belong beside the clone that was supposed to deliver it.

    Any of config / code / hardware missing raises `SystemExit` on the
    spot — responsibility 8 — so cell 1 never runs against a
    half-prepared runtime. `hardware_import` exists only so a test can
    drive `detect_hardware()`'s success path without a real GPU or a real
    `torch` install; the default is the real `importlib.import_module`.

    `check_accelerator()` runs LAST inside this `try`, strictly after
    `write_bootstrap_output()` — never before. Cell 1 already cannot run
    after this cell's own `SystemExit`, so "before training" is
    structural regardless of ordering; what ordering decides is whether
    the refusal's own evidence (the arriving device, the torch build, the
    installed arch list the verdict was computed from) is still readable
    on disk once it fires. It always is, because it was written first.
    """
    base = Path(base_dir) if base_dir is not None else Path.cwd()
    try:
        run_config = load_run_config(base)
        # S3/D6: the credential material is discovered by NAME at the
        # session-dir convention — `exec -f` has no argv/env channel, so
        # nothing can be injected through a process environment. Inert on
        # every credentialless run: a run-config with no staged askpass
        # beside it passes `None` and behaves exactly as before this
        # existed.
        askpass = base / REPO_CREDENTIAL_ASKPASS_FILENAME
        clone_dir = clone_repo(
            run_config,
            base / CLONE_DIRNAME,
            askpass=askpass if askpass.is_file() else None,
        )
        src_dir = add_clone_to_path(clone_dir)
        install_environment(run_config)
        modules = declared_modules(run_config)
        imports = verify_imports_under_clone(modules, src_dir)
        notebooks = verify_notebooks_under_clone(
            declared_notebooks(run_config), clone_dir
        )
        environment = detect_hardware(import_module=hardware_import)
        write_bootstrap_output(
            base,
            commit=run_config["commit"],
            run_config=run_config,
            environment=environment,
            imports=imports,
            notebooks=notebooks,
        )
        check_notebook_executor(notebooks)
        check_accelerator(run_config, environment)
    except BootstrapError as exc:
        raise SystemExit(f"bootstrap refused: {exc}") from exc
    return {
        "commit": run_config["commit"],
        "environment": environment,
        "imports": imports,
        "notebooks": notebooks,
    }


if __name__ == "__main__":
    bootstrap()


In [ ]:
#!/usr/bin/env python3
"""Cell 1 of the generated runner notebook — copied byte for byte.

Runs whichever of the two shapes the selected block declares.

- A CALLABLE block (`module` / `function` / `kwargs`) resolves through
  `importlib` and is called.
- A NOTEBOOK block (`notebook`) is read out of the clone, executed in a
  kernel with the clone's `src` on its `PYTHONPATH`, and written back
  executed into the runner's own working directory, which is what makes
  its outputs part of what a fetch returns.

Both stay reachable, deliberately. The notebook shape exists so that what
a worker runs can be the notebook a pilot validated, rather than a second
implementation of it whose agreement nothing checks. The callable shape
stays because a worker asked to settle one question is legitimately a
function call, and some entries cannot be named by a `module`/`function`
contract at all. `block_kind()` decides between them and refuses a block
that declares neither or both — never a default, because on this side of
the wire every default costs the same quota as the right answer.

The `smoke` variant is selected the same way for both shapes, when
`run_config["mode"] == "smoke"`. Cell 0
(`runner_bootstrap.py`) has already sparse-cloned the pinned commit and
put its `src/` on `sys.path` before this cell ever runs — `SystemExit`
there means this cell never runs at all — so the module named here
resolves from that same clone, never from anywhere this cell reaches on
its own initiative.

Importable and independently testable the same way `runner_bootstrap.py`
is: nothing runs at import time, and the orchestrating call sits behind
`if __name__ == "__main__":`, letting the forge suite drive
`select_block()`, `resolve_callable()` and `invoke()` directly against
fake `run-config.json` payloads.

No service name occurs anywhere in this file, and none may be added
later either.

Run with any Python 3.10+ (stdlib-only):
    python3 -m unittest tests.test_remote_execution
"""
from __future__ import annotations

import importlib
import json
import os
from pathlib import Path
from typing import Any, Callable, Mapping


class InvokeError(Exception):
    """A refusal: the selected `run`/`smoke` block, its declared module,
    or its declared function could not be resolved.
    """


CONFIG_FILENAME = "run-config.json"

# `runner_bootstrap.py` clones into `<base>/clone`, and this cell has to
# find the same directory to reach a declared notebook inside it. The two
# assets have no import between them — they are two cells of one notebook,
# each copied byte for byte — so this constant MIRRORS that one rather
# than importing it, exactly the way each file already holds its own
# subprocess-environment allowlist. A forge test binds the two spellings
# together; a divergence is a defect the suite catches, never a runtime
# surprise.
CLONE_DIRNAME = "clone"

# What an executed notebook is written back as, beside `run-config.json`
# and `bootstrap.json` in the runner's own working directory. A prefix and
# not an in-place overwrite: the clone is the pinned input and stays the
# pinned input, and the fetched artifact then carries both the notebook
# that was sent and the outputs it produced.
EXECUTED_NOTEBOOK_PREFIX = "executed-"

DEFAULT_KERNEL_NAME = "python3"

SRC_DIRNAME = "src"

# The handoff this cell owes the kernel it starts, beside the `PYTHONPATH`
# it already sets. Two names, forge-owned and deliberately generic — they
# are the contract between a runner and the notebook it starts, never a
# name borrowed from one repository — and mirrored, byte for byte, by
# `assets/notebook_repo_root.py`, the cell on the reading side. A forge
# test binds the two spellings together the way one already binds
# `CLONE_DIRNAME` to cell 0's.
#
# Why both, and never just the root: a directory handed over is still a
# directory nobody proved. The commit travels beside it so the reading
# cell can check the checkout it was pointed at rather than trust it, and
# a job that runs the wrong commit RUNS — it returns numbers shaped
# exactly like the right ones.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"


def kernel_python_path(clone_root: str | Path, existing: str | None) -> str:
    """`PYTHONPATH` for the kernel a notebook executes in — the clone's own
    `src`, ahead of whatever was already there.

    Cell 0's `sys.path.insert(0, <clone>/src)` puts the pinned code on the
    path of the RUNNER process, and a notebook executes in a SEPARATE
    kernel process that inherits none of it. Without this, a notebook
    whose first cell imports the target's own package dies with
    `ModuleNotFoundError` on a worker whose clone is sitting right there
    — the pinned code delivered, and unreachable.

    Prepended rather than replacing: an existing `PYTHONPATH` belongs to
    the worker's own image and this has no business discarding it. The
    clone goes first for the same reason cell 0 inserts at position 0 —
    a copy of the same package installed elsewhere must never win against
    the commit this job pinned.
    """
    clone_src = str((Path(clone_root) / SRC_DIRNAME).resolve())
    if existing:
        return clone_src + os.pathsep + existing
    return clone_src


def kernel_environment(
    clone_root: str | Path,
    commit: str,
    existing: Mapping[str, str] | None = None,
) -> dict:
    """Everything the kernel receives from this cell, composed in one place.

    Three variables, and this function is the only thing that decides them:
    the `PYTHONPATH` that makes the pinned code importable, and the two that
    tell the notebook WHERE the pinned code is and WHICH commit it is. The
    single composition point is the point — a second place that exported
    either of these would be a second place that decides which tree a
    notebook runs against, and the whole reason this handoff exists is that
    a notebook left to work that out on its own resolves a directory that
    exists on any worker and fails much later, naming a package rather than
    a root.

    `existing` is the environment being extended, so `PYTHONPATH` is
    prepended to rather than replaced; the two forge-owned names are set
    outright, because the only thing that could already be carrying them is
    an earlier runner and this one's clone is the one that counts.
    """
    existing = {} if existing is None else existing
    return {
        "PYTHONPATH": kernel_python_path(clone_root, existing.get("PYTHONPATH")),
        CLONE_ROOT_ENV: str(Path(clone_root).resolve()),
        CLONE_COMMIT_ENV: str(commit),
    }


def block_kind(block: Mapping[str, Any]) -> str:
    """Which of the two shapes a selected block declares — `"callable"`
    (a `module`/`function` pair) or `"notebook"` (a `notebook` path) —
    refusing anything that is neither and anything that is both.

    The mirror of `jobfolder.run_block_kind()`, held here on its own for
    the reason every guard in these two assets is: this cell runs against
    whatever `run-config.json` the kernel was actually handed, not against
    the one a generator validated. A block that declares neither shape is
    refused rather than defaulted, and a block that declares both is
    refused rather than resolved by precedence — a silent precedence rule
    would decide, on the worker, which of two disagreeing declarations the
    operator paid for.
    """
    if not isinstance(block, Mapping):
        raise InvokeError(f"selected block {block!r} is not an object")
    notebook = block.get("notebook")
    module = block.get("module")
    function = block.get("function")
    if notebook is not None and (module is not None or function is not None):
        raise InvokeError(
            f"selected block declares BOTH a notebook ({notebook!r}) and a "
            f"module/function pair ({module!r}/{function!r}); a block "
            "declares exactly one of the two"
        )
    if notebook is not None:
        if not isinstance(notebook, str) or not notebook.strip():
            raise InvokeError(
                f"selected block declares notebook {notebook!r}: a notebook "
                "path must be a non-empty string"
            )
        return "notebook"
    if module and function:
        return "callable"
    raise InvokeError(
        f"selected block {dict(block)!r} declares neither a "
        "'module'/'function' pair nor a 'notebook'"
    )


def select_block(run_config: Mapping[str, Any]) -> Mapping[str, Any]:
    """The normal `run` block, or its `smoke` variant when
    `run_config["mode"] == "smoke"` — T11 sets that mode; this is the
    branch it selects.
    """
    run_block = run_config.get("run", {})
    if run_config.get("mode") == "smoke":
        smoke = run_block.get("smoke")
        if not smoke:
            raise InvokeError(
                "run_config declares mode 'smoke' but its 'run' block "
                "carries no 'smoke' entry"
            )
        return smoke
    return run_block


def resolve_callable(
    block: Mapping[str, Any],
    *,
    import_module: Callable[[str], Any] = importlib.import_module,
) -> Callable[..., Any]:
    """Resolve `block["module"]`/`block["function"]` through `importlib`,
    refusing a missing module, a missing attribute, or a non-callable
    attribute — never guessing at any of the three.
    """
    module_name = block.get("module")
    function_name = block.get("function")
    if not module_name or not function_name:
        raise InvokeError(
            f"block {dict(block)!r} is missing a 'module' or a 'function'"
        )
    try:
        module = import_module(module_name)
    except ImportError as exc:
        raise InvokeError(f"could not import {module_name!r}: {exc}") from exc
    try:
        func = getattr(module, function_name)
    except AttributeError as exc:
        raise InvokeError(
            f"{module_name!r} has no attribute {function_name!r}"
        ) from exc
    if not callable(func):
        raise InvokeError(f"{module_name}.{function_name} is not callable")
    return func


def _default_notebook_client(notebook: Any, *, kernel_name: str, cwd: Path) -> Any:
    """The real executor, built only when a notebook run actually reaches
    it — never at import time, so the forge suite can drive every other
    function in this file on a machine that has no kernel at all.

    `nbclient` is what executes; `resources.metadata.path` is the working
    directory it starts the kernel in. That directory is the RUNNER's own
    working directory and deliberately not the notebook's location inside
    the clone: a notebook writing a relative path is writing an output,
    and an output written inside the clone is an output the fetch never
    sees. The clone is the pinned input; the working directory is what
    comes back.
    """
    from nbclient import NotebookClient

    return NotebookClient(
        notebook,
        kernel_name=kernel_name,
        resources={"metadata": {"path": str(cwd)}},
    )


def execute_notebook(
    block: Mapping[str, Any],
    base_dir: str | Path | None = None,
    *,
    commit: Any = None,
    client_factory: Callable[..., Any] = _default_notebook_client,
) -> dict:
    """Execute the declared notebook — the notebook half of cell 1.

    The notebook is read from the CLONE (`<base>/clone/<notebook>`, the
    pinned bytes cell 0 fetched), executed with the kernel it names, and
    written back — outputs and all — to `<base>/executed-<name>.ipynb`,
    the runner's own working directory. That destination is the whole of
    "the outputs come back": the fetch materializes the files the worker
    left in its working directory, which is already how `bootstrap.json`
    returns, so an executed notebook written there needs nothing added
    anywhere else in this skill to arrive.

    The kernel is started with `kernel_environment()`'s three variables:
    the clone's `src` first on `PYTHONPATH`, and the clone's directory and
    the pinned commit under the two forge-owned names the notebook's own
    first cell reads. That handoff is why `commit` is a parameter of this
    function at all — it comes from `run-config.json` by way of
    `invoke()`, and it is the only thing that lets the notebook check the
    checkout it is pointed at instead of trusting it.

    Four things this refuses rather than works around:

    - a run with no commit to hand over. This is the metered path; a
      notebook started without the pin is a notebook that has to work the
      repository out for itself, and the answer it works out is a
      directory that exists on any worker. Refused BEFORE the kernel
      starts, because a job that runs the wrong commit works;
    - a notebook path that escapes the clone, or is not a file inside it
      (cell 0 already proves this for a declared notebook, and this cell
      proves it again because it is the cell that opens the file);
    - a missing `nbformat`/`nbclient` — an `ImportError` becomes an
      `InvokeError` naming `environment.install` as the remedy, the same
      remedy cell 0's own executor gate names;
    - an execution that raises — the partially-executed notebook is
      written to the same destination BEFORE the failure is re-raised, so
      the fetched artifact carries the traceback in the cell that
      produced it instead of nothing at all.

    `client_factory` exists only so the forge suite can drive this
    function without a kernel; the default is the real one.
    """
    if not isinstance(commit, str) or not commit.strip():
        raise InvokeError(
            f"cannot start a notebook kernel: the pinned commit is {commit!r}, "
            f"so neither {CLONE_ROOT_ENV} nor {CLONE_COMMIT_ENV} can be handed "
            "over as a pair. Half a handoff is refused on the reading side too; "
            "a notebook left to locate the repository on its own resolves a "
            "directory that exists on any worker and fails much later, naming a "
            "package rather than a root"
        )
    base = Path(base_dir) if base_dir is not None else Path.cwd()
    name = block["notebook"]
    clone_root = (base / CLONE_DIRNAME).resolve()
    notebook_path = (clone_root / name).resolve()
    try:
        notebook_path.relative_to(clone_root)
    except ValueError:
        raise InvokeError(
            f"declared notebook {name!r} resolves to {notebook_path}, outside "
            f"the clone at {clone_root}"
        ) from None
    if not notebook_path.is_file():
        raise InvokeError(
            f"declared notebook {name!r} is not at {notebook_path}; the "
            "sparse checkout delivered nothing for it"
        )
    try:
        import nbformat
    except ImportError as exc:
        raise InvokeError(
            "cannot execute a notebook: 'nbformat' is not importable in this "
            "runtime. Declare it in run-config.json's environment.install "
            f"(--environment-requirement): {exc}"
        ) from exc

    notebook = nbformat.read(str(notebook_path), as_version=4)
    kernelspec = (notebook.get("metadata") or {}).get("kernelspec") or {}
    kernel_name = str(kernelspec.get("name") or DEFAULT_KERNEL_NAME)
    executed_path = base / f"{EXECUTED_NOTEBOOK_PREFIX}{notebook_path.name}"

    try:
        client = client_factory(notebook, kernel_name=kernel_name, cwd=base)
    except ImportError as exc:
        raise InvokeError(
            "cannot execute a notebook: 'nbclient' is not importable in this "
            "runtime. Declare it in run-config.json's environment.install "
            f"(--environment-requirement): {exc}"
        ) from exc

    handoff = kernel_environment(clone_root, commit, os.environ)
    saved = {variable: os.environ.get(variable) for variable in handoff}
    os.environ.update(handoff)
    try:
        client.execute()
    finally:
        for variable, previous in saved.items():
            if previous is None:
                os.environ.pop(variable, None)
            else:
                os.environ[variable] = previous
        # Written on the failure path too, and that is the point: a
        # notebook that died halfway is the only record of WHERE it died,
        # and losing it means paying the quota again to find out.
        nbformat.write(notebook, str(executed_path))

    return {
        "notebook": name,
        "kernel": kernel_name,
        "executed": str(executed_path),
        "cells": len(notebook.get("cells") or []),
    }


def invoke(
    run_config: Mapping[str, Any],
    *,
    import_module: Callable[[str], Any] = importlib.import_module,
    base_dir: str | Path | None = None,
    client_factory: Callable[..., Any] = _default_notebook_client,
) -> Any:
    """The whole of cell 1: select the block, decide which of the two
    shapes it declares, and run that one.

    A callable block resolves through `importlib` and is called with its
    declared kwargs, exactly as it always was. A notebook block is
    executed from the clone and written back executed. `block_kind()`
    decides, and it refuses a block that is neither or both — this
    function has no default branch, because the only default available
    costs the same quota as the right answer.

    `run_config["commit"]` is read here and handed down rather than looked
    up again lower: this is where the config is, and a pin re-read
    somewhere else would be a second place that decides which commit a
    notebook runs against. Read with `.get()` and never `[...]` — a
    missing pin becomes `execute_notebook()`'s own refusal, which says
    what could not be handed over, instead of a `KeyError` that says only
    that a dictionary was missing a key.
    """
    block = select_block(run_config)
    if block_kind(block) == "notebook":
        return execute_notebook(block, base_dir,
                                commit=run_config.get("commit"),
                                client_factory=client_factory)
    func = resolve_callable(block, import_module=import_module)
    kwargs = dict(block.get("kwargs") or {})
    return func(**kwargs)


def _load_run_config(base_dir: str | Path | None = None) -> dict:
    base = Path(base_dir) if base_dir is not None else Path.cwd()
    path = base / CONFIG_FILENAME
    if not path.is_file():
        raise InvokeError(f"{path} does not exist")
    return json.loads(path.read_text(encoding="utf-8"))


if __name__ == "__main__":
    invoke(_load_run_config())
